In [22]:
import langchain
import chromadb
import pypdf
import os

DOCS_DIR = r"../documents"
CHROMA_DIR = r"../chroma_db"

In [23]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
loader = PyPDFDirectoryLoader(DOCS_DIR)
documents = loader.load()

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap=100)
chunks = splitter.split_documents(documents)

In [25]:
import hashlib
def log_chunk_id(doc):
    source = doc.metadata.get("source", "")
    page = doc.metadata.get("page", "")
    h = hashlib.sha256(f"{source}-{page}-{doc.page_content}".encode()).hexdigest()
    return h

ids = [log_chunk_id(c) for c in chunks]

In [26]:
from langchain_voyageai import VoyageAIEmbeddings
from langchain_chroma import Chroma

embedding_model = VoyageAIEmbeddings(model="voyage-4-lite")
vector_store = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=embedding_model
)

In [27]:
# from langchain_experimental.text_splitter import SemanticChunker

# semantic_splitter = SemanticChunker(
#     embeddings=embedding_model,                       # note: 'embeddings', plural
#     breakpoint_threshold_type="gradient",             # note: '_type', not bare 'breakpoint_threshold'
#     breakpoint_threshold_amount=0.8, 
# )

# chunks = semantic_splitter.split_documents(documents)
# print(chunks[0])

In [28]:
# max = 0
# for chunk in chunks:
#     chunk_size = len(chunk.page_content)
#     if chunk_size > max:
#         max = chunk_size

# print(max)

In [29]:
existing = set(vector_store.get()["ids"])
new_chunks = [c for c, i in zip(chunks, ids) if i not in existing]
new_ids = [i for i in ids if i not in existing]

if new_chunks:
    vector_store.add_documents(documents=new_chunks, ids=new_ids)
    print(f"Added {len(new_chunks)} new chunks.")
else:
    print("No new chunks to add.")

No new chunks to add.


In [35]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)

In [36]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)

In [37]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Use the following pieces of context to answer the question at the end.
If you don't know the answer, say that you don't know.
Context: {context}
Question: {question}
""")

In [38]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [40]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

result = chain.invoke("Why did monster hunter world get so many players again in 2023?")
print(result)

Based on the provided context, *Monster Hunter: World* gained a large number of players again in December 2023 because Capcom announced *Monster Hunter Wilds* and launched a "Return to World" campaign to spur interest. This campaign and announcement led to the game rising to the top of simultaneous player counts on Steam, reaching over 100,000 concurrent players.
